# AviationStack flight API (from `.env`)

This notebook loads your **AviationStack** access key from `Week 4/.env` and calls the [**Flights**](https://aviationstack.com/documentation) endpoint.

**In `.env`, set either:**
- `AVIATIONSTACK_ACCESS_KEY=your_key_here` (recommended), or  
- keep using `MY_API_KEY=` if that value is already your AviationStack key.

The request uses `urllib` and the standard library only (no `requests` / `python-dotenv` required).

In [1]:
import json
import os
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path


def load_env(path: Path) -> None:
    """Load KEY=VALUE lines into os.environ (does not overwrite existing vars)."""
    if not path.exists():
        raise FileNotFoundError(f"Missing env file: {path}")
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        if "=" not in line:
            continue
        key, _, value = line.partition("=")
        key, value = key.strip(), value.strip().strip("'").strip('"')
        if key and key not in os.environ:
            os.environ[key] = value


env_path = Path(".env").resolve()
load_env(env_path)

access_key = os.environ.get("AVIATIONSTACK_ACCESS_KEY") or os.environ.get("MY_API_KEY")
if not access_key:
    raise ValueError(
        "Set AVIATIONSTACK_ACCESS_KEY (or MY_API_KEY) in .env next to this notebook."
    )
print("Loaded access key from .env (value not printed).")

Loaded access key from .env (value not printed).


## Call `/v1/flights`

Adjust **`params`** below (for example `flight_iata`, `dep_iata`, `flight_date`) to match what you want to query. On the free plan, very wide queries may be slow or hit limits; starting with a **small `limit`** is a good idea.

In [2]:
BASE = "https://api.aviationstack.com/v1/flights"

params = {
    "access_key": access_key,
    "limit": 10,
}

url = f"{BASE}?{urllib.parse.urlencode(params)}"
req = urllib.request.Request(url, headers={"Accept": "application/json"})

with urllib.request.urlopen(req, timeout=60) as resp:
    payload = json.loads(resp.read().decode("utf-8"))

if "error" in payload:
    err = payload["error"]
    raise RuntimeError(f"AviationStack error: {err}")

flights = payload.get("data") or []
print("Keys in response:", sorted(payload.keys()))
print("Number of flight objects:", len(flights))

Keys in response: ['data', 'pagination']
Number of flight objects: 10


## Inspect a few flights

Each item in `data` is nested (departure, arrival, airline, flight, …). Below we pull a few flat fields for a quick read.

In [4]:
rows = []
for item in flights[:10]:
    dep = item.get("departure") or {}
    arr = item.get("arrival") or {}
    airline = item.get("airline") or {}
    flight = item.get("flight") or {}
    rows.append(
        {
            "flight_date": item.get("flight_date"),
            "flight_status": item.get("flight_status"),
            "airline": airline.get("name"),
            "flight_iata": flight.get("iata"),
            "dep_airport": dep.get("iata"),
            "dep_scheduled": dep.get("scheduled"),
            "arr_airport": arr.get("iata"),
            "arr_scheduled": arr.get("scheduled"),
        }
    )

for r in rows:
    print(r)

{'flight_date': '2026-04-24', 'flight_status': 'scheduled', 'airline': 'empty', 'flight_iata': None, 'dep_airport': 'BHS', 'dep_scheduled': '2026-04-24T11:00:00+00:00', 'arr_airport': 'BHS', 'arr_scheduled': '2026-04-23T12:08:00+00:00'}
{'flight_date': '2026-04-24', 'flight_status': None, 'airline': 'empty', 'flight_iata': None, 'dep_airport': 'TSV', 'dep_scheduled': '2026-04-24T09:30:00+00:00', 'arr_airport': 'TSV', 'arr_scheduled': '2026-04-23T11:30:00+00:00'}
{'flight_date': '2026-04-23', 'flight_status': 'scheduled', 'airline': 'AirAsia', 'flight_iata': 'AK3025', 'dep_airport': 'DMK', 'dep_scheduled': '2026-04-23T10:30:00+00:00', 'arr_airport': 'HKT', 'arr_scheduled': '2026-04-23T12:00:00+00:00'}
{'flight_date': '2026-04-23', 'flight_status': 'scheduled', 'airline': 'Turkish Airlines', 'flight_iata': 'TK7586', 'dep_airport': 'HKT', 'dep_scheduled': '2026-04-23T14:45:00+00:00', 'arr_airport': 'BKK', 'arr_scheduled': '2026-04-23T16:20:00+00:00'}
{'flight_date': '2026-04-23', 'flight_

## Optional: full JSON for one record

Use this when you need to see every nested field AviationStack returns.

In [5]:
if flights:
    print(json.dumps(flights[0], indent=2, ensure_ascii=False)[:4000])
else:
    print("No flights in this response — try different query params or check your plan limits.")

{
  "flight_date": "2026-04-24",
  "flight_status": "scheduled",
  "departure": {
    "airport": "Raglan",
    "timezone": "Australia/Sydney",
    "iata": "BHS",
    "icao": "YBTH",
    "terminal": null,
    "gate": null,
    "delay": 72,
    "scheduled": "2026-04-24T11:00:00+00:00",
    "estimated": "2026-04-24T11:00:00+00:00",
    "actual": "2026-04-23T11:35:00+00:00",
    "estimated_runway": "2026-04-23T11:35:00+00:00",
    "actual_runway": "2026-04-23T11:35:00+00:00"
  },
  "arrival": {
    "airport": "Raglan",
    "timezone": "Australia/Sydney",
    "iata": "BHS",
    "icao": "YBTH",
    "terminal": null,
    "gate": null,
    "baggage": null,
    "scheduled": "2026-04-23T12:08:00+00:00",
    "delay": 71,
    "estimated": "2026-04-23T13:19:00+00:00",
    "actual": null,
    "estimated_runway": null,
    "actual_runway": null
  },
  "airline": {
    "name": "empty",
    "iata": null,
    "icao": null
  },
  "flight": {
    "number": null,
    "iata": null,
    "icao": null,
    "co